# Session 2 — Multi-Conformer UV/vis and ECD Workflow

**eChem School — Marseille 2026**

## Why do conformers matter for spectra?

Flexible molecules exist in solution as a **Boltzmann ensemble** of conformers.
Each conformer has its own geometry, excitation energies, and transition moments.
The observed spectrum is a **population-weighted average** over all contributing structures.

This is especially critical for **ECD**: different conformers can produce bands of
opposite sign, so ignoring the ensemble can give qualitatively wrong results.

The workflow in this session:

```
MM conformer search
        │
        ▼
Boltzmann filter (MM energies, cutoff p > 5 %)
        │
        ▼
DFT geometry optimization  +  single-point energy (high basis)
        │
        ▼
Boltzmann filter (QM energies)
        │
        ▼
TDDFT  (UV/vis + ECD)  for each surviving conformer
        │
        ▼
Boltzmann-weighted combined spectrum
```

## Step 1 — Load the molecule

We use **L-alanine** as the test molecule: small, chiral, flexible enough to have
several conformers with different Boltzmann weights.

In [ ]:
import veloxchem as vlx
import math
import numpy as np

molecule = vlx.Molecule.read_name('L-alanine')
molecule.show()

## Step 2 — Computational parameters

We use a **two-tier DFT strategy** that balances speed and accuracy:

| Stage | Functional | Basis set | Purpose |
|-------|-----------|-----------|---------|
| Optimization | B3LYP-D4 | def2-SVP | Fast relaxation of each conformer |
| Single-point | M06-L | def2-TZVP | Accurate relative energies for Boltzmann weights |
| TDDFT | CAM-B3LYP | def2-SVP | Excitation energies and ECD |

The **Boltzmann cutoff** (5 %) discards conformers whose population is negligible,
avoiding unnecessary expensive calculations.

In [ ]:
# Functionals and basis sets
opt_functional   = "B3LYP"
opt_basis        = "def2-svp"
sp_functional    = "m06-l"
sp_basis         = "def2-tzvp"
tddft_functional = "cam-b3lyp"
tddft_basis      = "def2-svp"

# Workflow settings
max_opt_steps           = 15
number_of_tddft_states  = 6

# Boltzmann settings
temp   = 298.15  # K
cutoff = 0.05    # discard conformers with weight < 5 %

## Step 3 — Generate MM conformers

`ConformerGenerator` performs a systematic torsional search using the MMFF94
force field and removes duplicates by RMSD.  This gives us a ranked list of
low-energy geometries to feed into the DFT pipeline.

In [ ]:
conf_gen = vlx.ConformerGenerator()
conformations = conf_gen.generate(molecule)

print(f"Number of conformers found: {len(conformations['molecules'])}")

In [ ]:
# Visualise the five lowest MM conformers
conf_gen.show_conformers(5)

## Step 4 — Boltzmann filter on MM energies

The Boltzmann weight of conformer $i$ at temperature $T$ is

$$
p_i = \frac{e^{-\Delta E_i / k_B T}}{\sum_j e^{-\Delta E_j / k_B T}},
\qquad \Delta E_i = E_i - E_0
$$

where energies are in kJ/mol and $k_B = 8.314 \times 10^{-3}\,\text{kJ mol}^{-1}\text{K}^{-1}$.

Conformers with $p_i < $ `cutoff` are discarded before the expensive DFT step.

In [ ]:
boltzmann_mm = []
for i in range(len(conformations['molecules'])):
    delta_e = conformations['energies'][i] - conformations['energies'][0]  # kJ/mol
    weight  = math.exp(-delta_e * 1000 / (8.314 * temp))
    boltzmann_mm.append(weight)

boltzmann_mm = np.array(boltzmann_mm) / np.sum(boltzmann_mm)

print("MM Boltzmann weights:")
for i, w in enumerate(boltzmann_mm):
    flag = " ✓" if w > cutoff else " ✗ (filtered)"
    print(f"  Conformer {i+1}: {w:.4f}{flag}")

## Step 5 — DFT optimization and single-point energies

For each conformer that passes the MM filter we:
1. **Optimize** the geometry at B3LYP-D4/def2-SVP (with RI-J approximation for speed)
2. Compute a **single-point energy** at M06-L/def2-TZVP on the optimized geometry

The resulting QM energies give much more reliable Boltzmann weights than the
MM energies from step 4.

In [ ]:
energies   = []
geometries = []

for i in range(len(conformations['molecules'])):
    if boltzmann_mm[i] <= cutoff:
        continue

    print(f"\n─── Conformer {i+1} (MM weight = {boltzmann_mm[i]:.3f}) ───")

    # ── DFT optimization ──────────────────────────────────────────────────
    basis   = vlx.MolecularBasis.read(conformations['molecules'][i], opt_basis)
    scf_opt = vlx.ScfRestrictedDriver()
    scf_opt.xcfun       = opt_functional
    scf_opt.dispersion  = True
    scf_opt.conv_thresh = 1e-3
    scf_opt.grid_level  = 2
    scf_opt.ri_coulomb  = True
    scf_opt.ostream.mute()
    results_opt = scf_opt.compute(conformations['molecules'][i], basis)

    opt_drv          = vlx.OptimizationDriver(scf_opt)
    opt_drv.max_iter = max_opt_steps
    opt_drv.conv_maxiter = True
    opt_results      = opt_drv.compute(conformations['molecules'][i], basis, results_opt)
    geometries.append(opt_results['final_geometry'])

    # ── Single-point energy ───────────────────────────────────────────────
    mol_opt  = vlx.Molecule.read_xyz_string(opt_results['final_geometry'])
    basis_sp = vlx.MolecularBasis.read(mol_opt, sp_basis)
    scf_sp   = vlx.ScfRestrictedDriver()
    scf_sp.xcfun       = sp_functional
    scf_sp.dispersion  = True
    scf_sp.conv_thresh = 1e-3
    scf_sp.grid_level  = 4
    scf_sp.ri_coulomb  = True
    scf_sp.ostream.mute()
    results_sp = scf_sp.compute(mol_opt, basis_sp)
    energies.append(results_sp['scf_energy'])

    print(f"  SP energy = {results_sp['scf_energy']:.8f} Hartree")

## Step 6 — Boltzmann weights from QM single-point energies

We repeat the Boltzmann analysis using the QM energies (in Hartree).
The conversion factor 2625.5 × 1000 converts Hartree → J/mol.

In [ ]:
boltzmann_qm = []
E0 = min(energies)
for E in energies:
    delta_e = (E - E0) * 2625.5 * 1000  # Hartree → J/mol
    boltzmann_qm.append(math.exp(-delta_e / (8.314 * temp)))

boltzmann_qm = np.array(boltzmann_qm) / np.sum(boltzmann_qm)

print("QM Boltzmann weights:")
for i, w in enumerate(boltzmann_qm):
    flag = " ✓" if w > cutoff else " ✗ (filtered)"
    print(f"  Conformer {i+1}: {w:.4f}{flag}")

## Step 7 — TDDFT for each surviving conformer

We run `LinearResponseEigenSolver` at CAM-B3LYP/def2-SVP to obtain:
- **Excitation energies** $\omega_{n0}$
- **Oscillator strengths** $f_{n0}$ → UV/vis intensity
- **Rotatory strengths** $R_{n0}$ → ECD sign and magnitude

We also request **natural transition orbitals (NTOs)** for visualization of the
dominant electron/hole pairs for each excitation.

In [ ]:
rpa_eigenvalues        = []
rpa_oscillator_strengths = []
rpa_rotatory_strengths = []

for i, geom in enumerate(geometries):
    if boltzmann_qm[i] <= cutoff:
        continue

    print(f"\n─── TDDFT for conformer {i+1} (QM weight = {boltzmann_qm[i]:.3f}) ───")

    mol   = vlx.Molecule.read_xyz_string(geom)
    basis = vlx.MolecularBasis.read(mol, tddft_basis)

    scf_drv          = vlx.ScfRestrictedDriver()
    scf_drv.xcfun    = tddft_functional
    scf_drv.filename = f"rpa_conf_{i+1}"
    scf_drv.ostream.mute()
    scf_results = scf_drv.compute(mol, basis)

    rpa_solver = vlx.LinearResponseEigenSolver()
    rpa_solver.nstates = number_of_tddft_states
    rpa_solver.nto     = True
    rpa_results = rpa_solver.compute(mol, basis, scf_results)

    rpa_eigenvalues.append(rpa_results['eigenvalues'])
    rpa_oscillator_strengths.append(rpa_results['oscillator_strengths'])
    rpa_rotatory_strengths.append(rpa_results['rotatory_strengths'])

## Step 8 — Boltzmann-weighted combined spectrum

The weighted oscillator and rotatory strengths are

$$
f_{n0}^\mathrm{avg} = \sum_i p_i\, f_{n0}^{(i)},
\qquad
R_{n0}^\mathrm{avg} = \sum_i p_i\, R_{n0}^{(i)}
$$

Concatenating all states (with their weighted intensities) into a single
`combined_rpa` dict lets us pass it directly to the VeloxChem plotting functions.

In [ ]:
# Select only the conformers that passed the QM Boltzmann filter
surviving = [i for i, w in enumerate(boltzmann_qm) if w > cutoff]

osc_weighted = [rpa_oscillator_strengths[k] * boltzmann_qm[surviving[k]]
                for k in range(len(surviving))]
rot_weighted = [rpa_rotatory_strengths[k]   * boltzmann_qm[surviving[k]]
                for k in range(len(surviving))]

combined_rpa = {
    'eigenvalues':         np.concatenate([rpa_eigenvalues[k]        for k in range(len(surviving))]),
    'oscillator_strengths': np.concatenate(osc_weighted),
    'rotatory_strengths':   np.concatenate(rot_weighted),
}

print(f"Combined spectrum built from {len(surviving)} conformers.")

## Step 9 — Plot: individual vs. Boltzmann-averaged spectra

Compare the spectrum of the **lowest-energy conformer** alone with the
**Boltzmann-weighted average** to see how the ensemble averaging shifts and
broadens both UV/vis and ECD.

In [ ]:
print("Boltzmann-weighted average (all surviving conformers):")
rpa_solver.plot_uv_vis(combined_rpa)
rpa_solver.plot_ecd(combined_rpa, broadening_value=0.3)

print("\nLowest-energy conformer only:")
single_conf = {
    'eigenvalues':          rpa_eigenvalues[0],
    'oscillator_strengths': rpa_oscillator_strengths[0],
    'rotatory_strengths':   rpa_rotatory_strengths[0],
}
rpa_solver.plot_uv_vis(single_conf)
rpa_solver.plot_ecd(single_conf, broadening_value=0.3)

## Summary and key takeaways

| Step | Tool | Purpose |
|------|------|---------|
| MM conformer search | `ConformerGenerator` | Fast enumeration of candidate geometries |
| MM Boltzmann filter | numpy | Discard high-energy structures early |
| DFT opt + SP | `OptimizationDriver` + `ScfRestrictedDriver` | Accurate geometries and relative energies |
| QM Boltzmann filter | numpy | Population weights for spectral averaging |
| TDDFT | `LinearResponseEigenSolver` | Excitation energies, oscillator & rotatory strengths |
| Weighted average | numpy | Boltzmann-averaged spectrum |

**Key messages:**

1. **Single-conformer spectra can be misleading.** For chiral flexible molecules,
   ECD bands from different conformers can cancel or amplify — the correct spectrum
   requires ensemble averaging.
2. **Two-tier DFT** (cheap opt, accurate SP) gives reliable Boltzmann weights at
   manageable cost.
3. **CAM-B3LYP** with range separation is preferred for UV/vis and ECD of organic
   molecules because it corrects the spurious low-lying CT states present in
   pure/hybrid GGAs.